## Jupyter Notebook — dARK + Authority Test Script

In [ ]:
import json
import time
import sys
import json
from pathlib import Path
from web3 import Web3, HTTPProvider
from eth_account import Account
import configparser
from web3.middleware import geth_poa_middleware

# ==========================================
# CONFIGURATION
# ==========================================

RPC_URL = "http://127.0.0.1:8545"

AUTHORIZED_ACCOUNT = "0xFE3B557E8Fb62b89F4916B721be55cEb828dBd73"
AUTHORIZED_PRIVATE_KEY = "0x8f2a55949038a9610f50fb23b5883af3b4ecb3c3bb792cbcefbd1542c692be63"

UNAUTHORIZED_ACCOUNT = "0x627306090abaB3A6e1400e9345bC60c78a8BEf57"
UNAUTHORIZED_PRIVATE_KEY = "0x..."

CHAIN_ID = 31337   # change if needed

# Connect to node
web3 = Web3(HTTPProvider(RPC_URL))
web3.middleware_onion.inject(geth_poa_middleware, layer=0)

CHAIN_ID = web3.eth.chain_id

print("Chain ID:", CHAIN_ID)




In [ ]:
# ==========================================
# PATH CONFIGURATION
# ==========================================

# Root project directory
PROJECT_ROOT = Path("../").resolve()

COMPILED_DIR = PROJECT_ROOT / "compiled"

print("Project root:", PROJECT_ROOT)
print("Compiled dir:", COMPILED_DIR)

In [ ]:
accounts = web3.eth.accounts

ADMIN = web3.to_checksum_address("0xFE3B557E8Fb62b89F4916B721be55cEb828dBd73")
AUTHORITY = web3.to_checksum_address("0x1111111111111111111111111111111111111111")
UNAUTHORIZED = web3.to_checksum_address("0x2222222222222222222222222222222222222222")

# print("Admin:", ADMIN)
# print("Authority:", AUTHORITY)
# print("Unauthorized:", UNAUTHORIZED)

In [ ]:
# ==========================================
# LOAD DEPLOYED CONTRACTS
# ==========================================

config = configparser.ConfigParser()

config.read("../deployed_contracts.ini")

authority_address = web3.to_checksum_address(
    config["Authority"]["address"]
)

authority_abi = json.loads(
    config["Authority"]["abi"]
)

dark_address = web3.to_checksum_address(
    config["dARK"]["address"]
)

dark_abi = json.loads(
    config["dARK"]["abi"]
)

print("Authority:", authority_address)
print("dARK:", dark_address)

In [ ]:
# ==========================================
# CREATE CONTRACT INSTANCES
# ==========================================

authority = web3.eth.contract(
    address=authority_address,
    abi=authority_abi
)

dark = web3.eth.contract(
    address=dark_address,
    abi=dark_abi
)

print("Contracts loaded successfully")

In [ ]:
# ==========================================
# CHECK INITIAL STATE
# ==========================================

admin = authority.functions.admin().call()

print("Authority admin:", admin)

In [ ]:
# ==========================================
# REGISTER AUTHORITY
# ==========================================

uuid = "authority-001"
encrypted_key = "encrypted-key-placeholder"

# Build transaction
txn = authority.functions.register_authority(
    uuid,
    AUTHORITY,
    encrypted_key
).build_transaction({
    "from": ADMIN,
    "nonce": web3.eth.get_transaction_count(ADMIN),
    "gas": 200000,
    "gasPrice": web3.to_wei("40", "gwei"),
    "chainId": web3.eth.chain_id
})

# Sign transaction
signed_txn = web3.eth.account.sign_transaction(txn, AUTHORIZED_PRIVATE_KEY)

# Send raw transaction
tx_hash = web3.eth.send_raw_transaction(signed_txn.rawTransaction)

# Wait for receipt
receipt = web3.eth.wait_for_transaction_receipt(tx_hash)

print("Authority registered, tx hash:", tx_hash.hex())

In [ ]:
# ==========================================
# AUTHORIZE NAAN
# ==========================================

naan = "12345"

# 1️⃣ Build transaction
txn = authority.functions.authorize_naan(
    naan
).build_transaction({
    "from": AUTHORITY,
    "nonce": web3.eth.get_transaction_count(AUTHORITY),
    "gas": 200_000,
    "gasPrice": web3.to_wei("40", "gwei"),
    "chainId": web3.eth.chain_id
})

# 2️⃣ Sign transaction with private key of AUTHORITY account
signed_txn = web3.eth.account.sign_transaction(txn, AUTHORITY)

# 3️⃣ Send signed transaction
tx_hash = web3.eth.send_raw_transaction(signed_txn.rawTransaction)

# 4️⃣ Wait for transaction receipt
receipt = web3.eth.wait_for_transaction_receipt(tx_hash)

print(f"NAAN '{naan}' authorized, tx hash: {tx_hash.hex()}")

In [ ]:
is_auth = authority.functions.is_authorized(
    AUTHORITY,
    naan
).call()

print("Authorized:", is_auth)

In [ ]:
tx = dark.functions.create_ark(
    naan,
    "my-first-ark",
    "https://example.com",
    "QmCID123"
).transact({
    "from": AUTHORITY
})

web3.eth.wait_for_transaction_receipt(tx)

print("ARK created")

In [ ]:
url = dark.functions.resolve(
    naan,
    "my-first-ark"
).call()

print("Resolved URL:", url)

## FAIL TEST (unauthorized)

In [ ]:
try:

    tx = dark.functions.create_ark(
        "99999",
        "unauthorized-ark",
        "https://fail.com",
        "CIDFAIL"
    ).transact({
        "from": UNAUTHORIZED
    })

    web3.eth.wait_for_transaction_receipt(tx)

except Exception as e:

    print("Expected failure:")
    print(e)

In [ ]:
## Check ARK exists
exists = dark.functions.ark_exists(
    naan,
    "my-first-ark"
).call()

print("ARK exists:", exists)

In [ ]:
## Get full ARK
ark = dark.functions.get_ark(
    naan,
    "my-first-ark"
).call()

print("ARK Data:")
print("Name:", ark[0])
print("NAAN:", ark[1])
print("URL:", ark[2])
print("CID:", ark[3])
print("Owner:", ark[4])
print("Created:", ark[5])
print("Updated:", ark[6])